# CRYPTO SCALPER LAB v0.8.2 — L2 Conditioning + Maker Fill Lab
Prospective public Binance L2 + trade capture. Frozen thresholds are written before capture. No API key, no live trading.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, shutil, pathlib
REPO='/content/crypto-scalper-lab'
if os.path.exists(REPO): shutil.rmtree(REPO)
!git clone --depth 1 --branch v0.8.2-l2-maker-lab https://github.com/bognar-aron/bognar-aron-crypto-scalper-lab.git {REPO}
%cd {REPO}
!pip -q install -r requirements.txt
!pip -q install -r requirements-v08.txt


In [ ]:
from datetime import datetime, timezone
import json
from l2_conditioning_lab_v082 import PROTOCOL
ROOT=pathlib.Path('/content/drive/MyDrive/CRYPTO_SCALPER_LAB_v04')
RUNS=ROOT/'runs'
RUNS.mkdir(parents=True, exist_ok=True)
stamp=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
OUT=RUNS/f'v082_L2_{stamp}'
CAPTURE=OUT/'capture'
OUT.mkdir(parents=True, exist_ok=True)
(OUT/'v082_protocol_PRECAPTURE.json').write_text(json.dumps(PROTOCOL, indent=2), encoding='utf-8')
print('Output:', OUT)
print('Protocol frozen before capture.')


## Prospective capture
Default: BTC/USDC, ETH/USDC, SOL/USDC, 30 minutes, depth 10, 100 ms feature snapshots. Keep the tab/runtime alive while this cell runs.

In [ ]:
CAPTURE_SECONDS=1800
!python microstructure_lab_v08.py capture --symbols BTC-USDC,ETH-USDC,SOL-USDC --seconds {CAPTURE_SECONDS} --depth 10 --snapshot-interval-ms 100 --out "{CAPTURE}"


## Frozen conditioning + conservative shadow maker fills
The model gives no cancellation credit and treats all visible L1 size as queue ahead. It measures one-sided maker-entry fill and post-fill markout, not full round-trip PnL.

In [ ]:
!python l2_conditioning_lab_v082.py --capture-dir "{CAPTURE}" --out "{OUT}"


In [ ]:
import pandas as pd
display(pd.read_csv(OUT/'l2_condition_diagnostics.csv'))
display(pd.read_csv(OUT/'maker_fill_summary.csv'))
print(json.dumps(json.loads((OUT/'v082_gate.json').read_text()), indent=2))
